# TP 1 : ingérer un CSV réel, les accidents corporels de 2024

**Durée : 35 minutes.**

Les données sont celles de l'Observatoire national interministériel de la sécurité routière (ONISR), publiées sur
[data.gouv.fr](https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024)
sous Licence Ouverte. Chaque année comporte quatre fichiers reliés par l'identifiant d'accident `Num_Acc` :
caractéristiques, lieux, véhicules et usagers. La description des colonnes est sur la page du jeu de données.

À la fin du TP, vous saurez :
- déposer un fichier dans la zone brute sans le retélécharger à chaque exécution ;
- lire un CSV avec des paramètres explicites, et vérifier que les types lus sont les bons ;
- écrire des contrôles d'import qui refusent un fichier défectueux ;
- mesurer et réduire le temps de lecture et la mémoire occupée.

Après chaque exercice, une cellule `assert` vérifie votre résultat. Si elle ne lève pas d'erreur, passez à la suite.
Tant qu'un exercice n'est pas rempli, il s'arrête sur le message « Exercice à compléter ».

In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests

RAW = Path("data/raw/accidents_2024")
CLEAN = Path("data/clean/accidents")
TMP = Path("data/tmp")
TMP.mkdir(parents=True, exist_ok=True)

BASE = (
    "https://static.data.gouv.fr/resources/bases-de-donnees-annuelles-des-accidents-corporels-de-la-"
    "circulation-routiere-annees-de-2005-a-2024"
)
URL_CARACT = f"{BASE}/20251021-115900/caract-2024.csv"
print("pandas", pd.__version__)


# Appelée par les cellules d'exercice tant que vous ne les avez pas remplies
def a_completer():
    raise NotImplementedError("Exercice à compléter : remplacez a_completer() par votre code")

## Partie 1 : déposer le fichier dans la zone brute (5 min)

Écrivez `telecharger(url, dest)` :
- si `dest` existe déjà, ne rien faire et renvoyer `dest` : relancer le notebook ne doit pas retélécharger ;
- sinon, télécharger avec `requests.get`, **avec un `timeout`**, et lever une erreur si le code HTTP n'est pas 200
  (`raise_for_status`) ;
- créer le dossier si besoin, écrire le contenu, renvoyer `dest`.

Une copie des quatre fichiers est déjà dans `data/raw/accidents_2024` : la fonction ne retéléchargera donc rien là.
Pour vérifier qu'elle télécharge vraiment, on l'appelle sur un dossier de test.

In [ ]:
def telecharger(url: str, dest: Path, timeout: int = 60) -> Path:
    a_completer()  # À compléter

In [ ]:
try:
    essai = telecharger(URL_CARACT, TMP / "telechargement" / "caract-2024.csv")
    assert essai.stat().st_size == (RAW / "caract-2024.csv").stat().st_size
    print("Téléchargement vérifié :", essai)
except requests.RequestException as erreur:
    print("Pas de réseau, on continue avec la copie locale :", erreur)

assert telecharger(URL_CARACT, RAW / "caract-2024.csv") == RAW / "caract-2024.csv"

## Partie 2 : lire les caractéristiques des accidents (12 min)

### 2.1 La lecture naïve

In [ ]:
try:
    pd.read_csv(RAW / "caract-2024.csv")
except Exception as erreur:
    print(type(erreur).__name__, ":", erreur)

Avant de changer un paramètre au hasard, on regarde le fichier brut.

In [ ]:
print((RAW / "caract-2024.csv").read_text(encoding="utf-8")[:500])

**Question.** Pourquoi la lecture naïve échoue-t-elle à la ligne 618 seulement, et pas dès la première ?

*Votre réponse :*

### 2.2 Le séparateur seul ne suffit pas

In [ ]:
pd.read_csv(RAW / "caract-2024.csv", sep=";").dtypes

**Question.** Citez trois colonnes dont le type lu ne convient pas, et dites pourquoi.

*Votre réponse :*

### 2.3 La lecture explicite

Complétez `lire_caract` pour que :
- `lat` et `long` soient des nombres (`decimal`) ;
- `Num_Acc`, `dep`, `com`, `hrmn` et `adr` soient du texte (`dtype`) ;
- les codes `lum`, `agg`, `int`, `atm` et `col` soient des entiers `Int8`, qui acceptent les valeurs manquantes ;
- `-1` soit lu comme une valeur manquante (`na_values`), qu'il soit écrit `-1` ou avec une espace devant.

In [ ]:
CODES_CARACT = ["lum", "agg", "int", "atm", "col"]
NON_RENSEIGNE = ["-1", " -1"]


def lire_caract(path: Path) -> pd.DataFrame:
    # À compléter : decimal, encoding, na_values, dtype
    return pd.read_csv(path, sep=";")


caract = lire_caract(RAW / "caract-2024.csv")
caract.dtypes

In [ ]:
assert caract["lat"].dtype == "float64", "lat doit être un nombre : pensez à decimal"
assert caract["Num_Acc"].dtype == "str", "un identifiant se lit en texte"
assert caract["dep"].dtype == "str" and caract["com"].dtype == "str"
assert str(caract["lum"].dtype) == "Int8"
assert caract["col"].isna().sum() == 6, "-1 signifie non renseigné : pensez à na_values"
print(len(caract), "accidents lus correctement")

### 2.4 Une vraie date

Écrivez `ajouter_date_heure(df)`, qui ajoute une colonne `date_heure` de type `datetime64` à partir de `an`, `mois`,
`jour` et `hrmn` (format `07:40`), puis supprime ces quatre colonnes. Donnez un `format` explicite à
`pd.to_datetime` : sans lui, pandas devine, et peut confondre jour et mois.

La fonction s'enchaîne ensuite avec `pipe` : chaque étape reste une fonction courte, testable seule.

In [ ]:
def ajouter_date_heure(df: pd.DataFrame) -> pd.DataFrame:
    a_completer()  # À compléter


caract = lire_caract(RAW / "caract-2024.csv").pipe(ajouter_date_heure)
caract[["Num_Acc", "date_heure"]].head()

In [ ]:
assert str(caract["date_heure"].dtype).startswith("datetime64")
assert caract["date_heure"].min() >= pd.Timestamp("2024-01-01")
assert caract["date_heure"].max() < pd.Timestamp("2025-01-01")
assert "hrmn" not in caract.columns

## Partie 3 : les usagers, et le lien avec les accidents (6 min)

Complétez `lire_usagers` : identifiants en texte, `actp` en `category`, `an_nais` en `Int16`, les autres codes en
`Int8`, et `-1` comme valeur manquante. Regardez ensuite `id_usager` : à quoi servent les espaces ?

In [ ]:
CODES_USAGERS = ["place", "catu", "grav", "sexe", "trajet", "secu1", "secu2", "secu3", "locp", "etatp"]
TYPES_USAGERS = {}  # À compléter


def lire_usagers(path: Path) -> pd.DataFrame:
    a_completer()  # À compléter


usagers = lire_usagers(RAW / "usagers-2024.csv")
usagers.head(3)

In [ ]:
assert usagers["Num_Acc"].dtype == "str"
assert str(usagers["grav"].dtype) == "Int8" and str(usagers["an_nais"].dtype) == "Int16"
print(repr(usagers["id_usager"].iloc[0]))

*Votre réponse :*

Chaque usager doit se rattacher à un accident connu. Calculez le nombre d'usagers orphelins, puis le nombre moyen
d'usagers par accident.

In [ ]:
orphelins = a_completer()  # À compléter
par_accident = a_completer()  # À compléter
print(orphelins, "orphelins ;", round(par_accident, 2), "usagers par accident en moyenne")

In [ ]:
assert orphelins == 0
assert 2 < par_accident < 3

## Partie 4 : les contrôles d'import (4 min)

Un fichier défectueux doit arrêter l'ingestion avec un message clair, au lieu de produire un résultat faux en
silence. Complétez `check_import`, qui applique les cinq contrôles d'import :
1. le fichier existe ;
2. il n'est pas vide ;
3. les colonnes attendues sont présentes, après lecture avec la fonction `lire` ;
4. le nombre de lignes est plausible ;
5. les types lus commencent par le type attendu, par exemple `"float"` pour `float64`.

Chaque échec lève une `ErreurImport`. La qualité des valeurs (domaines, doublons) viendra en séance 3.

In [ ]:
class ErreurImport(Exception):
    pass


def check_import(path: Path, lire, colonnes: list[str], min_lignes: int, types: dict[str, str]) -> pd.DataFrame:
    # À compléter : les cinq contrôles, dans l'ordre, puis renvoyer le DataFrame lu
    a_completer()

On fabrique trois fichiers défectueux à partir du vrai, puis on vérifie que chacun est refusé.

In [ ]:
lignes = (RAW / "caract-2024.csv").read_text(encoding="utf-8").splitlines()
(TMP / "vide.csv").write_text("", encoding="utf-8")
renommee = [lignes[0].replace('"lat"', '"latitude"')] + lignes[1:]
(TMP / "colonne_renommee.csv").write_text("\n".join(renommee), encoding="utf-8")
(TMP / "tronque.csv").write_text("\n".join(lignes[:101]), encoding="utf-8")

lire = lambda path: lire_caract(path).pipe(ajouter_date_heure)  # noqa: E731
COLONNES = ["Num_Acc", "dep", "com", "lat", "long", "date_heure"]
TYPES = {"dep": "str", "lat": "float", "long": "float", "date_heure": "datetime64"}


def est_refuse(path: Path) -> bool:
    try:
        check_import(path, lire, COLONNES, 40_000, TYPES)
    except ErreurImport as erreur:
        print("Refusé :", erreur)
        return True
    return False


assert all(est_refuse(TMP / nom) for nom in ["absent.csv", "vide.csv", "colonne_renommee.csv", "tronque.csv"])
assert len(check_import(RAW / "caract-2024.csv", lire, COLONNES, 40_000, TYPES)) == 54_402

## Partie 5 : optimiser la vitesse et la mémoire (10 min)

On travaille sur le plus gros fichier, les usagers. Deux outils de mesure sont fournis : `mesurer` garde le meilleur
de trois essais, `memoire` compte la mémoire réelle, contenu des chaînes compris (`deep=True`).

In [ ]:
USAGERS = RAW / "usagers-2024.csv"
UTILES = ["Num_Acc", "id_vehicule", "catu", "grav", "sexe", "an_nais", "trajet", "secu1"]


def mesurer(fonction, essais: int = 3) -> float:
    temps = []
    for _ in range(essais):
        debut = time.perf_counter()
        fonction()
        temps.append(time.perf_counter() - debut)
    return min(temps)


def memoire(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1e6

Complétez les trois lectures manquantes :
- **explicite** : celle de `lire_usagers` ;
- **colonnes utiles** : seulement les colonnes de `UTILES` (`usecols`), avec leurs types de `TYPES_USAGERS` ;
- **pyarrow** : la lecture explicite, avec `engine="pyarrow"`.

In [ ]:
lectures = {
    "implicite (sep seul)": lambda: pd.read_csv(USAGERS, sep=";"),
    "explicite": lambda: a_completer(),  # À compléter
    "colonnes utiles": lambda: a_completer(),  # À compléter
    "explicite, pyarrow": lambda: a_completer(),  # À compléter
}

In [ ]:
resultats = pd.DataFrame(
    [{"lecture": nom, "temps_s": mesurer(lire), "memoire_mo": memoire(lire())} for nom, lire in lectures.items()]
).set_index("lecture")
resultats.round(3)

Une fois le CSV lu correctement, on ne le relit plus : on l'écrit en Parquet, un format en colonnes, compressé, qui
conserve les types. Écrivez `usagers` en Parquet dans `TMP`, puis comparez la taille des deux fichiers et le temps de
relecture.

In [ ]:
parquet = TMP / "usagers.parquet"
# À compléter : écrire, comparer les tailles, puis ajouter la ligne « relecture Parquet » à resultats
a_completer()
resultats.round(3)

In [ ]:
assert resultats.loc["colonnes utiles", "memoire_mo"] < resultats.loc["implicite (sep seul)", "memoire_mo"] / 2
assert resultats.loc["explicite, pyarrow", "temps_s"] < resultats.loc["explicite", "temps_s"]
assert pd.read_parquet(parquet)["an_nais"].dtype == usagers["an_nais"].dtype, "Parquet conserve les types"

**Question.** La lecture explicite est-elle plus rapide que la lecture implicite ? Qu'en concluez-vous ?

*Votre réponse :*

## Écrire le résultat

Les fichiers propres vont dans `data/clean`. Le TP 2 en a besoin.

In [ ]:
CLEAN.mkdir(parents=True, exist_ok=True)
caract.to_parquet(CLEAN / "caract.parquet", index=False)
usagers.to_parquet(CLEAN / "usagers.parquet", index=False)
sorted(p.name for p in CLEAN.iterdir())

## Bonus 1 : la même lecture avec Polars

Mesurez `pl.read_csv(USAGERS, separator=";")` avec `mesurer`, puis regardez le schéma : Polars devine-t-il mieux les
types que pandas ?

In [ ]:
import polars as pl

# À compléter

*Votre réponse :*

## Bonus 2 : le fichier des lieux

Lisez `lieux-2024.csv`. Trois pièges vous attendent : la valeur `#VALEURMULTI` dans `nbv`, la virgule décimale de
`larrout`, et des points de repère `pr` écrits `3 900`, avec une espace insécable. Essayez `thousands="\xa0"` :
que constatez-vous ?

In [ ]:
# À compléter